# Sprint 2 — Del análisis exploratorio al primer modelo de Machine Learning

**Dataset:** [House price prediction (Kaggle — `shree1992/housedata`)](https://www.kaggle.com/datasets/shree1992/housedata)

**Entorno:** Google Colab — continuación del trabajo iniciado en el **Sprint 1**.

---

## Índice

1. **Fase 1 — Análisis exploratorio y visual, preparación de datos y primer modelo de regresión**
2. **Fase 2 — Interpretación propia de los hallazgos**
3. **Fase 3 — Uso de IA generativa como apoyo al análisis**
4. **Fase 4 — Comparación crítica entre la interpretación propia y la generada por la IA**

> **Objetivo del cuaderno:** partir del dataset de viviendas importado en el Sprint 1,
> profundizar en el análisis exploratorio, preparar los datos y entrenar un **primer
> modelo de regresión** para predecir el precio (`price`). Después, interpretar los
> resultados de forma propia, contrastarlos con una IA generativa y comparar
> críticamente ambas interpretaciones.

## Configuración inicial

Importamos las librerías de análisis (`pandas`, `numpy`, `matplotlib`, `seaborn`) y
las de modelado (`scikit-learn`). Todas vienen preinstaladas en Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("Librerías importadas correctamente")

### Carga del dataset desde Kaggle

En el Sprint 1 documentamos tres vías de importación (local, API de Kaggle y Google
Drive). Aquí usamos **`kagglehub`**, la librería oficial de Kaggle preinstalada en
Colab: para datasets **públicos** no requiere token, descarga el dataset a una caché
local y devuelve la ruta, por lo que el cuaderno es totalmente reproducible.

Como respaldo, si la descarga fallara (p. ej. sin conexión), intentamos leer
la copia local descargada en el Sprint 1.

In [ ]:
import os

try:
    import kagglehub
    ruta = kagglehub.dataset_download("shree1992/housedata")
    print("Dataset descargado de Kaggle en:", ruta)
    df = pd.read_csv(os.path.join(ruta, "data.csv"))
except Exception as e:
    print("No se pudo descargar de Kaggle:", e)
    print("Usando la copia local del Sprint 1...")
    df = pd.read_csv("../sprint1/data/data.csv")

print("Dimensiones (filas, columnas):", df.shape)
df.head()

---
# Fase 1 — Análisis exploratorio y visual, preparación de datos y primer modelo

**Objetivo:** profundizar en el EDA iniciado en el Sprint 1 con un enfoque ya
orientado al modelado: estudiar la **variable objetivo** (`price`), las
**distribuciones** de las variables, las **relaciones** entre ellas y el objetivo,
limpiar los datos y entrenar un **primer modelo de regresión** interpretable.

## 1.1 — La variable objetivo: `price`

Antes de nada estudiamos la variable que queremos predecir. Su forma condiciona
todas las decisiones posteriores (tratamiento de outliers, posible transformación
logarítmica, métrica de error...).

In [ ]:
print("Estadísticos de price:")
print(df["price"].describe().round(0))
print("\nAsimetría (skewness):", round(df["price"].skew(), 2))
print("Viviendas con price = 0:", (df["price"] == 0).sum())
print("Precio máximo:", f"{df['price'].max():,.0f}")
print("Percentil 99:", f"{df['price'].quantile(0.99):,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["price"], bins=60, kde=True, ax=axes[0])
axes[0].set_title("Distribución de price (escala original)")
axes[0].set_xlabel("Precio ($)")

precios_validos = df.loc[df["price"] > 0, "price"]
sns.histplot(np.log10(precios_validos), bins=60, kde=True, ax=axes[1], color="orange")
axes[1].set_title("Distribución de price (log10, precios > 0)")
axes[1].set_xlabel("log10(Precio)")

plt.tight_layout()
plt.show()

**Lectura:** la distribución del precio está **fuertemente sesgada a la derecha**
(asimetría ≈ 1,7 incluso tras limpiar; media ≈ \$528k frente a mediana ≈ \$461k) y
contiene valores claramente problemáticos:

- **49 viviendas con `price = 0`**: precio no informado, no son ventas reales utilizables.
- Una cola de **outliers extremos** (máximo \$26,6M frente a un percentil 99 de ≈ \$2,0M)
  que distorsionaría un modelo lineal entrenado por mínimos cuadrados.

En escala logarítmica la distribución se vuelve aproximadamente simétrica, lo que
anticipa que una transformación log podría beneficiar al modelo (lo comprobaremos
empíricamente en la Fase 4).

## 1.2 — Limpieza y preparación del dataset

Aplicamos las decisiones de limpieza que dejamos planteadas en el Sprint 1,
justificando cada una:

| Decisión | Justificación |
|---|---|
| Eliminar `price = 0` (49 filas) | Registros sin precio informado: no aportan señal y romperían la escala log. |
| Eliminar `bedrooms = 0` o `bathrooms = 0` (4 filas) | Muy probablemente errores de registro. |
| Eliminar precios por encima del **percentil 99** (≈ \$2,0M, 46 filas) | Outliers extremos (hasta \$26,6M) con enorme influencia sobre una regresión lineal; son un mercado distinto (ultra-lujo) al que no pretendemos modelar. |
| Descartar `date`, `street`, `country`, `statezip` | `date` cubre solo ~2 meses de 2014 (sin variación estacional aprovechable); `street` es casi única por vivienda; `country` es constante (`USA`); `statezip` (77 niveles) duplica la información de `city` (44 niveles) con más fragmentación. |
| Crear `antiguedad`, `renovada`, `tiene_sotano` | Variables derivadas más interpretables que `yr_built`, `yr_renovated` y `sqft_basement` en bruto. |

En total perdemos ~2% de las filas, un coste muy razonable a cambio de un dataset fiable.

In [ ]:
n0 = len(df)

# 1) Precios no informados
df_clean = df[df["price"] > 0].copy()
n1 = len(df_clean)

# 2) Posibles errores de registro
df_clean = df_clean[(df_clean["bedrooms"] > 0) & (df_clean["bathrooms"] > 0)]
n2 = len(df_clean)

# 3) Outliers extremos de precio (por encima del percentil 99)
p99 = df_clean["price"].quantile(0.99)
df_clean = df_clean[df_clean["price"] <= p99]
n3 = len(df_clean)

# 4) Variables derivadas (el dataset es de 2014)
df_clean["antiguedad"] = 2014 - df_clean["yr_built"]
df_clean["renovada"] = (df_clean["yr_renovated"] > 0).astype(int)
df_clean["tiene_sotano"] = (df_clean["sqft_basement"] > 0).astype(int)

print(f"Filas originales:                {n0}")
print(f"Tras eliminar price = 0:         {n1}  (-{n0 - n1})")
print(f"Tras eliminar 0 dorm./baños:     {n2}  (-{n1 - n2})")
print(f"Tras eliminar price > p99:       {n3}  (-{n2 - n3})   [p99 = {p99:,.0f}]")
print(f"\nPérdida total: {n0 - n3} filas ({(n0 - n3) / n0:.1%})")
df_clean[["price", "sqft_living", "antiguedad", "renovada", "tiene_sotano"]].head()

## 1.3 — Distribuciones de las variables explicativas

Revisamos la forma de las principales variables numéricas: nos interesa detectar
sesgos, escalas muy distintas y variables casi constantes.

In [ ]:
cols_dist = ["sqft_living", "sqft_lot", "sqft_above", "sqft_basement",
             "bedrooms", "bathrooms", "floors", "antiguedad"]

df_clean[cols_dist].hist(bins=40, figsize=(14, 8), edgecolor="black")
plt.suptitle("Distribución de las principales variables explicativas")
plt.tight_layout()
plt.show()

print("Frecuencia de las variables binarias / ordinales:")
for c in ["waterfront", "view", "condition", "renovada", "tiene_sotano"]:
    print()
    print(df_clean[c].value_counts().sort_index().to_string())

**Lectura:** las superficies (`sqft_living`, `sqft_lot`) también están sesgadas a la
derecha, algo esperable en variables de tamaño. Dos avisos importantes para la
interpretación posterior:

- `waterfront = 1` solo tiene **22 viviendas** (~0,5% del dataset): cualquier
  conclusión sobre ella tendrá mucha incertidumbre.
- `view > 0` y `condition ∈ {1, 2}` también son minoritarias: las medianas por grupo
  deben leerse con cuidado en los grupos pequeños.

## 1.4 — Relaciones entre variables y con el precio

Buscamos qué variables se relacionan más con `price` (candidatas a predictoras) y
qué variables están muy relacionadas **entre sí** (riesgo de multicolinealidad).

In [ ]:
plt.figure(figsize=(12, 9))
corr = df_clean.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            annot_kws={"size": 8})
plt.title("Matriz de correlación (variables numéricas)")
plt.tight_layout()
plt.show()

In [ ]:
correlacion_precio = (
    df_clean.select_dtypes(include=np.number)
    .corr()["price"]
    .drop("price")
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
correlacion_precio.plot(kind="barh", color=np.where(correlacion_precio > 0, "#4C72B0", "#C44E52"))
plt.title("Correlación de cada variable con el precio")
plt.xlabel("Coeficiente de correlación de Pearson")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(correlacion_precio.round(3))

In [ ]:
# Relaciones bivariadas clave con el precio
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.scatterplot(data=df_clean, x="sqft_living", y="price", alpha=0.25, ax=axes[0, 0])
axes[0, 0].set_title("Superficie habitable vs precio (r ≈ 0,68)")

sns.boxplot(data=df_clean, x="view", y="price", ax=axes[0, 1])
axes[0, 1].set_title("Calidad de las vistas vs precio")

sns.boxplot(data=df_clean, x="waterfront", y="price", ax=axes[1, 0])
axes[1, 0].set_title("Frente al agua (0/1) vs precio  —  ¡solo 22 casos con 1!")

sns.boxplot(data=df_clean, x="condition", y="price", ax=axes[1, 1])
axes[1, 1].set_title("Estado de conservación vs precio")

plt.tight_layout()
plt.show()

In [ ]:
# Medianas por grupo: cuantificamos lo que muestran los boxplots
print("Mediana de precio según waterfront:")
print(df_clean.groupby("waterfront")["price"].agg(["median", "count"]).round(0).to_string())

print("\nMediana de precio según view:")
print(df_clean.groupby("view")["price"].agg(["median", "count"]).round(0).to_string())

print("\nMediana de precio según condition:")
print(df_clean.groupby("condition")["price"].agg(["median", "count"]).round(0).to_string())

print("\nMediana de precio según renovada:")
print(df_clean.groupby("renovada")["price"].agg(["median", "count"]).round(0).to_string())

## 1.5 — La dimensión geográfica: precio por ciudad

`city` es la única variable categórica que conservamos. Comprobamos si la ubicación
discrimina precios, porque de ser así deberá entrar en el modelo (codificada con
*one-hot encoding*).

In [ ]:
precio_ciudad = (
    df_clean.groupby("city")["price"]
    .agg(mediana="median", n="count")
    .query("n >= 20")                      # solo ciudades con muestra suficiente
    .sort_values("mediana", ascending=False)
)

plt.figure(figsize=(10, 8))
sns.barplot(x=precio_ciudad["mediana"], y=precio_ciudad.index, color="#4C72B0")
plt.title("Mediana de precio por ciudad (ciudades con ≥ 20 viviendas)")
plt.xlabel("Mediana de precio ($)")
plt.tight_layout()
plt.show()

print("Top 3 ciudades más caras:")
print(precio_ciudad.head(3).round(0).to_string())
print("\nTop 3 ciudades más baratas:")
print(precio_ciudad.tail(3).round(0).to_string())

**Lectura:** la ubicación discrimina muchísimo: la mediana de **Mercer Island
(≈ \$944k)** casi **cuadruplica** la de **SeaTac (≈ \$239k)**. La ciudad tiene que
estar en el modelo.

## 1.6 — Preparación final y primer modelo de regresión

Entrenamos una **regresión lineal múltiple** como primer modelo: es el modelo más
interpretable (cada coeficiente se lee como "dólares adicionales por unidad de la
variable, manteniendo el resto constante"), lo que encaja con el objetivo del sprint
de **interpretar**, no solo predecir.

Estrategia de evaluación: separamos un **20% de test** (con semilla fija para que
sea reproducible) y comparamos dos versiones:

- **Modelo A** — solo características físicas de la vivienda.
- **Modelo B** — características físicas **+ ciudad** (one-hot).

La diferencia entre ambos nos dirá cuánto aporta la ubicación.

In [ ]:
features_fisicas = ["sqft_living", "bathrooms", "bedrooms", "floors", "waterfront",
                    "view", "condition", "sqft_above", "sqft_basement",
                    "antiguedad", "renovada"]

y = df_clean["price"]

# Modelo A: solo características físicas
X_a = df_clean[features_fisicas]

# Modelo B: físicas + ciudad codificada con one-hot
X_b = pd.get_dummies(df_clean[features_fisicas + ["city"]], columns=["city"], drop_first=True)

Xa_train, Xa_test, y_train, y_test = train_test_split(X_a, y, test_size=0.2, random_state=42)
Xb_train, Xb_test, _, _ = train_test_split(X_b, y, test_size=0.2, random_state=42)

print(f"Train: {len(Xa_train)} filas | Test: {len(Xa_test)} filas")
print(f"Modelo A: {Xa_train.shape[1]} variables | Modelo B: {Xb_train.shape[1]} variables")

In [ ]:
def evaluar(nombre, modelo, X_tr, X_te):
    modelo.fit(X_tr, y_train)
    pred = modelo.predict(X_te)
    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    print(f"{nombre:<35} R² = {r2:.3f} | MAE = ${mae:,.0f} | RMSE = ${rmse:,.0f}")
    return modelo, pred

modelo_a, pred_a = evaluar("Modelo A (físicas)", LinearRegression(), Xa_train, Xa_test)
modelo_b, pred_b = evaluar("Modelo B (físicas + ciudad)", LinearRegression(), Xb_train, Xb_test)

In [ ]:
# Coeficientes del Modelo A: el efecto de cada característica física
coefs = pd.Series(modelo_a.coef_, index=features_fisicas).sort_values(ascending=False)
print("Coeficientes del Modelo A (interpretación: $ adicionales por unidad,")
print("manteniendo el resto de variables constantes):\n")
print(coefs.round(1).to_string())
print(f"\nIntercepto: {modelo_a.intercept_:,.0f}")

In [ ]:
# Diagnóstico visual del Modelo B: predicción vs realidad y residuos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, pred_b, alpha=0.3)
lims = [0, y_test.max()]
axes[0].plot(lims, lims, "r--", label="Predicción perfecta")
axes[0].set_xlabel("Precio real ($)")
axes[0].set_ylabel("Precio predicho ($)")
axes[0].set_title("Modelo B: predicción vs realidad (test)")
axes[0].legend()

residuos = y_test - pred_b
axes[1].scatter(pred_b, residuos, alpha=0.3)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Precio predicho ($)")
axes[1].set_ylabel("Residuo (real - predicho)")
axes[1].set_title("Modelo B: residuos")

plt.tight_layout()
plt.show()

print(f"Mediana del precio en test: ${y_test.median():,.0f}")
print(f"MAE del Modelo B como % de la mediana: {mean_absolute_error(y_test, pred_b) / y_test.median():.1%}")

---
# Fase 2 — Interpretación propia de los hallazgos

A continuación recojo los principales hallazgos del análisis, **argumentados con los
números obtenidos** en la Fase 1. (Los valores exactos pueden variar mínimamente
entre ejecuciones, pero las magnitudes y conclusiones se mantienen.)

## Hallazgo 1 — La superficie habitable es el factor individual más importante

`sqft_living` es la variable más correlacionada con el precio (**r ≈ 0,68**), por
encima de baños (0,52) y muy por encima de dormitorios (0,34). Esto **confirma la
hipótesis 1 del Sprint 1**: el mercado paga por *espacio*, no por *número de
habitaciones*. El scatter de 1.4 muestra además que la relación es aproximadamente
lineal aunque con varianza creciente: a más superficie, más dispersión de precios,
señal de que en las viviendas grandes pesan otros factores (ubicación, calidades).

## Hallazgo 2 — La ubicación importa tanto como toda la física de la vivienda

El Modelo A (solo características físicas) alcanza **R² ≈ 0,51**; al añadir
únicamente la ciudad, el Modelo B sube a **R² ≈ 0,66** y el error medio cae de
≈ \$143k a ≈ \$107k (**−25%**). Es decir, *dónde* está la vivienda explica una parte
del precio comparable a *cómo* es la vivienda. El análisis por ciudad lo hace
tangible: la mediana de Mercer Island (≈ \$944k) casi cuadruplica la de SeaTac
(≈ \$239k). Esto **confirma la hipótesis 4 del Sprint 1**.

## Hallazgo 3 — La paradoja de los dormitorios: correlación positiva, coeficiente negativo

`bedrooms` correlaciona positivamente con el precio (r ≈ 0,34), pero su coeficiente
en la regresión es **negativo (≈ −\$36k por dormitorio)**. No es un error: la
correlación mide el efecto *bruto* (las casas con más dormitorios suelen ser más
grandes y por eso más caras), mientras que el coeficiente mide el efecto *a
superficie constante*: si dividimos los mismos metros en más dormitorios, salen
habitaciones más pequeñas, y eso el mercado lo penaliza. Es el ejemplo perfecto de
por qué **correlación y efecto causal/parcial no son lo mismo**, y de la
**multicolinealidad** entre `bedrooms`, `bathrooms` y `sqft_living`.

## Hallazgo 4 — Vistas y frente al agua tienen prima clara, pero con poca evidencia

La mediana de las viviendas frente al agua (≈ \$949k) **duplica** la del resto
(≈ \$460k), y la prima crece monótonamente con la calidad de las vistas (de ≈ \$443k
con `view = 0` a ≈ \$989k con `view = 4`). Esto apoya la hipótesis 2 del Sprint 1,
**pero** con una cautela empírica importante: solo hay **22 viviendas** con
`waterfront = 1` (~0,5% de la muestra), así que la estimación de esa prima es muy
imprecisa y no debería extrapolarse.

## Hallazgo 5 — Resultados contraintuitivos: renovación y estado de conservación

Dos resultados van contra la intuición y conviene señalarlos en vez de esconderlos:

- Las viviendas **renovadas tienen mediana inferior** (≈ \$445k) a las no renovadas
  (≈ \$480k) y `renovada` correlaciona *negativamente* con el precio (r ≈ −0,05).
  La explicación más plausible es que **se renuevan las casas antiguas**: la
  renovación es un marcador de antigüedad, no una mejora neta observable. La
  hipótesis 3 del Sprint 1 ("las renovadas se venden más caras") **no se confirma**
  en su forma simple.
- `condition` apenas correlaciona con el precio (r ≈ 0,06) y sus medianas no son
  monótonas, aunque los niveles 1–2 tienen tan pocas viviendas que esas medianas son
  poco fiables.

## Hallazgo 6 — Capacidad y límites del primer modelo

El Modelo B explica **dos tercios de la varianza** (R² ≈ 0,66) con un error medio de
≈ \$107k, un **23% de la mediana** del precio. Como primer modelo interpretable es
razonable, pero los residuos muestran **heterocedasticidad** (el error crece con el
precio): el modelo se equivoca más, y tiende a quedarse corto, en las viviendas
caras. Esto apunta a dos mejoras naturales: modelar el **logaritmo del precio** y
usar modelos no lineales — exactamente lo que contrastaremos con la IA en las fases
siguientes.

---
# Fase 3 — Uso de IA generativa como apoyo al análisis

## Metodología

Utilicé **Claude (Anthropic)** como modelo de IA generativa. Diseñé las preguntas
siguiendo dos criterios:

1. **Darle contexto cuantitativo real** (resultados concretos de mi análisis), porque
   la IA no puede ver los datos: si no se le dan los números, solo puede responder
   generalidades sobre datasets de viviendas.
2. **Pedirle cosas distintas en cada prompt**: explicar (P1), resolver una duda
   técnica concreta (P2) y proponer mejoras accionables (P3).

## Prompt 1 — Validación de la interpretación

> *"Estoy analizando un dataset de 4.600 viviendas del área de Seattle (2014) para
> predecir el precio. Tras limpiar precios a 0 y outliers (percentil 99), una regresión
> lineal con características físicas da R² = 0,51 y al añadir la ciudad (one-hot) sube a
> R² = 0,66 con MAE ≈ \$107k. Las correlaciones más altas con el precio son sqft_living
> (0,68), sqft_above (0,58) y bathrooms (0,52); waterfront tiene mediana doble pero solo
> 22 casos. ¿Es correcta mi interpretación de que la superficie y la ubicación son los
> dos factores dominantes? ¿Qué matices se me pueden estar escapando?"*

**Respuesta de la IA (resumen):** confirmó la interpretación general y añadió matices:
(a) el salto de R² al añadir la ciudad es de los más altos documentados en este tipo de
datasets, lo que sugiere que una granularidad geográfica aún mayor (código postal,
coordenadas) podría aportar más; (b) con 22 casos de `waterfront`, el intervalo de
confianza de su prima es enorme y un solo caso atípico puede mover la mediana — sugirió
no reportarla sin acompañarla del tamaño muestral; (c) advirtió que `sqft_living`,
`sqft_above` y `sqft_basement` son **linealmente dependientes** (la primera es
prácticamente la suma de las otras dos), por lo que sus coeficientes individuales no son
interpretables por separado; (d) recomendó complementar el MAE con el **error porcentual**
(MAPE o MAE/mediana) porque un error de \$107k no significa lo mismo en una casa de
\$250k que en una de \$900k.

## Prompt 2 — Duda técnica concreta

> *"En mi regresión, bedrooms correlaciona +0,34 con el precio pero su coeficiente sale
> negativo (≈ −\$36k por dormitorio) cuando sqft_living está en el modelo. ¿Cuál es la
> explicación estadística y cómo debería contarlo en un informe?"*

**Respuesta de la IA (resumen):** explicó que es un caso clásico de **efecto parcial con
multicolinealidad**: el coeficiente mide el efecto de añadir un dormitorio *manteniendo
fija la superficie*, es decir, compartimentar más un mismo espacio, lo que reduce el
tamaño medio de las habitaciones y resta valor. No es contradictorio con la correlación
positiva, que mezcla el efecto del dormitorio con el de la superficie que suele venir con
él. Para el informe sugirió la fórmula: *"a igualdad de superficie, más dormitorios
significa habitaciones más pequeñas"*. Advirtió además de que con multicolinealidad los
coeficientes son inestables (pueden cambiar de signo entre muestras) y sugirió comprobar
el **VIF** o eliminar una de las variables redundantes.

## Prompt 3 — Sugerencias de mejora

> *"¿Qué tres mejoras priorizarías sobre este análisis y modelo, teniendo en cuenta que
> el objetivo es didáctico (primer modelo) pero quiero rigor? Dame mejoras concretas y
> el porqué de cada una."*

**Respuesta de la IA (resumen):** propuso, por orden de prioridad:

1. **Modelar `log(price)` en lugar de `price`**: el precio es log-normal; con la escala
   log los errores se vuelven relativos (porcentuales), se corrige la
   heterocedasticidad y los outliers pierden influencia.
2. **Validación cruzada (k-fold) en lugar de un único split 80/20**: con ~4.500 filas,
   un único split hace que las métricas dependan del azar de la partición; con 5-fold
   se obtiene media ± desviación de cada métrica.
3. **Probar un modelo no lineal de contraste** (Random Forest o Gradient Boosting):
   no para sustituir al lineal, sino para **medir cuánta señal no lineal** queda sin
   capturar; si el no lineal no mejora mucho, el lineal es defendible.

También mencionó enriquecer la geografía (lat/long por código postal) y crear ratios
(precio por ft², baños por dormitorio) como ideas adicionales.

---
# Fase 4 — Comparación crítica: mi interpretación vs la de la IA

## 4.1 — Verificación empírica de las sugerencias de la IA

Antes de comparar, hago lo que la propia metodología del sprint pide: **no aceptar
las sugerencias de la IA sin contrastarlas**. Pruebo las dos más relevantes
(objetivo en log y modelo no lineal de contraste) sobre el mismo split.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Sugerencia 1 de la IA: modelar log(price)
modelo_log = LinearRegression().fit(Xb_train, np.log1p(y_train))
pred_log = np.expm1(modelo_log.predict(Xb_test))
print(f"{'Modelo B (referencia)':<40} R² = {r2_score(y_test, pred_b):.3f} | MAE = ${mean_absolute_error(y_test, pred_b):,.0f}")
print(f"{'Modelo B + log(price)':<40} R² = {r2_score(y_test, pred_log):.3f} | MAE = ${mean_absolute_error(y_test, pred_log):,.0f}")

# Sugerencia 3 de la IA: modelo no lineal de contraste
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(Xb_train, y_train)
pred_rf = rf.predict(Xb_test)
print(f"{'Random Forest (contraste no lineal)':<40} R² = {r2_score(y_test, pred_rf):.3f} | MAE = ${mean_absolute_error(y_test, pred_rf):,.0f}")

importancias = pd.Series(rf.feature_importances_, index=Xb_train.columns).sort_values(ascending=False)
print("\nTop 8 variables más importantes según el Random Forest:")
print(importancias.head(8).round(3).to_string())

**Resultado de la verificación:** la transformación log **mejora el MAE** (de ≈ \$107k
a ≈ \$102k) aunque no el R², coherente con lo que prometía la IA (errores más
*relativos*). El Random Forest, en cambio, **no supera** a la regresión lineal
(R² ≈ 0,66 en ambos): apenas queda señal no lineal que capturar con estas variables,
lo que —tal y como dijo la IA en el Prompt 3— **hace defendible el modelo lineal**.
Además, sus importancias (sqft_living ≈ 0,52, muy por delante del resto) coinciden
con mi Hallazgo 1, dándole robustez por una vía independiente.

## 4.2 — Coincidencias, diferencias y aportaciones

| Aspecto | Mi interpretación | La IA | Valoración |
|---|---|---|---|
| Factores dominantes | Superficie y ubicación (Hallazgos 1 y 2) | Coincide | **Coincidencia** plena, y el RF la confirma por tercera vía. |
| Paradoja de `bedrooms` | Efecto parcial negativo por multicolinealidad (Hallazgo 3) | Misma explicación, más precisa: coeficientes inestables, comprobar VIF | **Coincidencia + aportación**: la IA añadió el riesgo de inestabilidad y una herramienta concreta (VIF) que yo no había considerado. |
| `waterfront` | Prima clara pero solo 22 casos (Hallazgo 4) | Coincide y va más allá: no reportar la prima sin el tamaño muestral | **Coincidencia + buena práctica** de comunicación que adopto. |
| Dependencia lineal `sqft_living = sqft_above + sqft_basement` | No lo había detectado explícitamente | Lo señaló | **Aportación de la IA**: explica por qué los coeficientes de las tres superficies no deben interpretarse por separado. |
| Transformación log | La intuí por la asimetría (Hallazgo 6) | La priorizó con argumentos (errores relativos, heterocedasticidad) | **Coincidencia**, y la verificación empírica (4.1) le da la razón en el MAE. |
| Validación cruzada | No la apliqué (un solo split) | La señaló como prioridad 2 | **Aportación pendiente**: es la mejora metodológica más relevante que dejo para el siguiente sprint. |
| Renovadas más baratas (Hallazgo 5) | Lo detecté en los datos y propuse explicación (renovación como marcador de antigüedad) | No lo mencionó hasta que yo lo aporté | **Límite de la IA**: no ve los datos; los hallazgos inesperados los tiene que encontrar el analista. |

## 4.3 — Reflexión crítica sobre el uso de la IA

- **Lo que aportó:** profundidad estadística (VIF, inestabilidad de coeficientes,
  dependencia lineal entre superficies), priorización razonada de mejoras y mejores
  prácticas de comunicación de resultados. Funcionó como un **revisor experto**.
- **Lo que no puede hacer:** no ve los datos. Todo lo que sabe del dataset es lo que
  yo le conté; si mis números hubieran sido erróneos, habría razonado igual de
  convincentemente sobre cifras falsas. Los hallazgos *inesperados* (Hallazgo 5)
  salieron de explorar los datos, no de preguntarle.
- **Riesgo principal:** la fluidez de sus respuestas invita a aceptarlas sin
  comprobarlas. Por eso la verificación de 4.1 es la parte más importante de esta
  fase: una de sus sugerencias mejoró el modelo (log → MAE −5%) y otra resultó
  innecesaria para ganar rendimiento (RF no supera al lineal), algo que **solo se
  sabe ejecutando el experimento**, no preguntando.
- **Conclusión sobre el flujo de trabajo:** la combinación óptima es
  *analista explora y encuentra → IA revisa, explica y sugiere → analista verifica
  empíricamente*. La IA complementa el criterio propio; no lo sustituye.

---

## Conclusión del Sprint 2

Partiendo del dataset del Sprint 1, hemos realizado un EDA orientado al modelado,
limpiado el dataset (precios a 0, errores de registro, outliers > p99), entrenado un
**primer modelo de regresión interpretable** (R² ≈ 0,66; MAE ≈ \$102k–107k, un ~23%
de la mediana), extraído seis hallazgos argumentados empíricamente —incluida la
confirmación de dos hipótesis del Sprint 1, el rechazo de una y una paradoja
estadística instructiva—, contrastado el análisis con una IA generativa mediante
prompts con contexto cuantitativo, y **verificado empíricamente** sus sugerencias
antes de aceptarlas. Quedan como siguientes pasos la validación cruzada y el
enriquecimiento geográfico del modelo.